# 2. From pandas to portfolio decisions
A portfolio manager has two books that own some of the same stocks. The questions are: **where is the money concentrated, what is the unrealized P&L, and does a proposed trade reduce the concentration?**

We will apply the selection, joins, and aggregation from [Pandas basics](2_pandas_basics.ipynb). A book is a named group of holdings. Each row below is an acquisition lot: shares bought at one entry price. Combining lots is necessary before assessing a stock's total weight.

All quantities, prices, and limits are synthetic examples. Prices are in dollars; the portfolio starts with no cash or leverage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Value the lots with a current quote
The same ticker can appear in different books and at different acquisition costs. The quote table has one row per ticker. A many-to-one join lets every lot receive its quote without multiplying holdings.

In [ ]:
positions = pd.DataFrame({
    'lot_id': ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8'],
    'book': ['Core', 'Tactical', 'Core', 'Tactical', 'Core', 'Tactical', 'Core', 'Tactical'],
    'ticker': ['AAPL', 'AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM', 'TLT', 'MSFT'],
    'shares': [120, 40, 30, 50, 80, 100, 60, 20],
    'entry_price': [175.0, 180.0, 390.0, 110.0, 190.0, 105.0, 95.0, 395.0],
})
quotes = pd.DataFrame({
    'ticker': ['TLT', 'JPM', 'AAPL', 'XOM', 'NVDA', 'MSFT'],
    'price': [92.0, 198.0, 185.0, 112.0, 125.0, 420.0],
    'sector': ['Treasuries', 'Financials', 'Technology', 'Energy', 'Technology', 'Technology'],
})
display(positions)
display(quotes)

In [ ]:
valued = positions.merge(quotes, on='ticker', how='left', validate='many_to_one')
assert len(valued) == len(positions)
assert valued[['price', 'sector']].notna().all().all(), 'Resolve missing quote coverage before valuing the portfolio.'
valued['cost_basis'] = valued['shares'] * valued['entry_price']
valued['market_value'] = valued['shares'] * valued['price']
valued['unrealized_pnl'] = valued['market_value'] - valued['cost_basis']
portfolio_value = valued['market_value'].sum()
display(valued[['lot_id', 'book', 'ticker', 'cost_basis', 'market_value', 'unrealized_pnl']])
print(f'Portfolio value: ${portfolio_value:,.0f}')

## 2. Aggregate before interpreting a weight or cost
AAPL appears in two lots. Its weighted entry price is total cost divided by total shares, not the simple average of the two entry prices.

Unrealized P&L is current value minus acquisition cost. Return on cost divides that P&L by cost; portfolio weight divides current value by the whole portfolio value. These denominators answer different questions.

In [ ]:
asset_summary = valued.groupby('ticker').agg(
    lots=('lot_id', 'size'), shares=('shares', 'sum'), cost_basis=('cost_basis', 'sum'),
    market_value=('market_value', 'sum'), unrealized_pnl=('unrealized_pnl', 'sum'))
asset_summary['weighted_entry_price'] = asset_summary['cost_basis'] / asset_summary['shares']
asset_summary['weight_pct'] = asset_summary['market_value'] / portfolio_value * 100
asset_summary['return_on_cost_pct'] = asset_summary['unrealized_pnl'] / asset_summary['cost_basis'] * 100
display(asset_summary.sort_values('weight_pct', ascending=False).round(2))
print('AAPL: simple average entry price =', positions.loc[positions['ticker'] == 'AAPL', 'entry_price'].mean())
print('AAPL: share-weighted entry price =', asset_summary.loc['AAPL', 'weighted_entry_price'])
assert np.isclose(asset_summary['market_value'].sum(), portfolio_value)

### A holding has different weights in its book and in the portfolio
`groupby(...).transform('sum')` repeats each book's total on that book's original rows. Unlike an aggregation, it keeps one result per lot. That lets us calculate book weights without a separate merge.

The Core and Tactical weights each sum to 100%; summing all book weights together would give 200%.

In [ ]:
valued['book_total'] = valued.groupby('book')['market_value'].transform('sum')
valued['weight_in_book_pct'] = valued['market_value'] / valued['book_total'] * 100
display(valued[['lot_id', 'book', 'ticker', 'market_value', 'weight_in_book_pct']].round(2))
display(valued.groupby('book')['weight_in_book_pct'].sum())

## 3. Turn exposure into a decision table
For this case, the manager's limits are **25% in one ticker** and **55% in Technology**. These are example portfolio rules, not general risk recommendations.

Count distinct tickers as well as lots. Splitting one stock into multiple lots does not diversify the exposure.

In [ ]:
ticker_limit = 25.0
technology_limit = 55.0
sector_summary = valued.groupby('sector').agg(
    distinct_tickers=('ticker', 'nunique'), lots=('lot_id', 'size'), market_value=('market_value', 'sum'))
sector_summary['weight_pct'] = sector_summary['market_value'] / portfolio_value * 100
breaches = asset_summary.loc[asset_summary['weight_pct'] > ticker_limit,
                             ['shares', 'market_value', 'weight_pct']]
display(sector_summary.sort_values('weight_pct', ascending=False).round(2))
display(breaches.round(2))
print('Technology exceeds its limit:', sector_summary.loc['Technology', 'weight_pct'] > technology_limit)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
sector_summary['weight_pct'].sort_values().plot.barh(ax=axes[0], title='Where is portfolio value concentrated?')
axes[0].set_xlabel('Portfolio weight (%)')
asset_summary['unrealized_pnl'].sort_values().plot.barh(ax=axes[1], title='Which holdings contribute to unrealized P&L?')
axes[1].axvline(0, color='black', linewidth=0.7)
axes[1].set_xlabel('Unrealized P&L ($)')
plt.show()

## 4. A proposed rebalance can fix one limit and miss another
Consider selling **40 AAPL shares** and buying **20 JPM shares** at the quoted prices, with no fees or price movement. The unused proceeds become cash.

Calculate the result on an aggregated copy. This is an exposure scenario; allocating the sale to acquisition lots would be needed to calculate realized P&L and is a different question.

In [ ]:
scenario = asset_summary[['shares']].join(quotes.set_index('ticker')[['price', 'sector']], validate='one_to_one')
scenario.loc['AAPL', 'shares'] -= 40
scenario.loc['JPM', 'shares'] += 20
scenario_cash = 40 * scenario.loc['AAPL', 'price'] - 20 * scenario.loc['JPM', 'price']
assert scenario['shares'].ge(0).all() and scenario_cash >= 0
scenario['market_value'] = scenario['shares'] * scenario['price']
scenario_total = scenario['market_value'].sum() + scenario_cash
scenario['weight_pct'] = scenario['market_value'] / scenario_total * 100
scenario_sector_weights = scenario.groupby('sector')['market_value'].sum() / scenario_total * 100
assert np.isclose(scenario_total, portfolio_value)
display(scenario[['shares', 'market_value', 'weight_pct']].round(2))
print(f'Cash: ${scenario_cash:,.0f} ({scenario_cash / scenario_total:.2%} of the portfolio)')
print('AAPL within limit:', scenario.loc['AAPL', 'weight_pct'] <= ticker_limit)
print('Technology within limit:', scenario_sector_weights.loc['Technology'] <= technology_limit)

AAPL falls below 25%, but Technology remains slightly above 55%. Cash belongs in the portfolio denominator: excluding it would overstate every invested holding's weight.

A useful desk update distinguishes the improvement from the remaining breach. The analysis also depends on trustworthy identifiers and prices, which we inspect in the next notebook.

Next: [Visualize and clean stock data](4_visualize_and_clean.ipynb).

## Exercises
Apply the workflow to a changed delivery and a different proposed trade. Keep the original inputs intact.

### Your turn: a revised acquisition lot
The desk corrects lot L2 from 40 shares to **70 shares**, leaving its entry price unchanged. Use the corrected copy below and the existing quotes to calculate `your_asset_summary`, indexed by ticker, with `shares`, `cost_basis`, `market_value`, `unrealized_pnl`, and `weighted_entry_price`.

State AAPL's revised weighted entry price in `your_cost_note` and explain why it is not the average of the two entry prices.

In [ ]:
practice_positions = positions.copy()
practice_positions.loc[practice_positions['lot_id'] == 'L2', 'shares'] = 70
your_asset_summary = None
your_cost_note = ''

In [ ]:
if your_asset_summary is None:
    print('Value the corrected lots, aggregate by ticker, and calculate the share-weighted entry price.')
else:
    assert set(your_asset_summary.index) == set(quotes['ticker'])
    assert your_asset_summary.loc['AAPL', 'shares'] == 190
    assert np.isclose(your_asset_summary.loc['AAPL', 'cost_basis'], 33600)
    assert np.isclose(your_asset_summary.loc['AAPL', 'market_value'], 35150)
    assert np.isclose(your_asset_summary.loc['AAPL', 'unrealized_pnl'], 1550)
    assert np.isclose(your_asset_summary.loc['AAPL', 'weighted_entry_price'], 33600 / 190)
    assert positions.loc[positions['lot_id'] == 'L2', 'shares'].item() == 40
    assert your_cost_note.strip()
    print('Revised-lot checks passed. Explain how the larger lot changes the weighted average.')

### Your turn: test another rebalance
Return to the original portfolio. Sell **42 AAPL shares** and buy **20 XOM shares**, retaining the unused proceeds as cash. Ignore fees and price changes.

Create `your_scenario`, indexed by ticker, with `shares`, `price`, `sector`, `market_value`, and `weight_pct`. Store the cash balance in `your_cash` and explain in `your_scenario_note` whether both limits are met. Include cash in the total portfolio value.

**Stretch:** find the smallest whole number of AAPL shares that must be sold to meet the Technology limit, keeping all proceeds as cash.

In [ ]:
your_scenario = None
your_cash = None
your_scenario_note = ''

In [ ]:
if your_scenario is None or your_cash is None:
    print('Apply the trade on a copy, retain cash, and check both exposure limits.')
else:
    assert your_scenario.loc['AAPL', 'shares'] == 118
    assert your_scenario.loc['XOM', 'shares'] == 120
    assert np.isclose(your_cash, 5530)
    assert np.isclose(your_scenario['market_value'].sum() + your_cash, portfolio_value)
    np.testing.assert_allclose(your_scenario['weight_pct'], your_scenario['market_value'] / portfolio_value * 100)
    assert your_scenario['weight_pct'].max() <= ticker_limit
    your_technology_weight = your_scenario.loc[your_scenario['sector'] == 'Technology', 'weight_pct'].sum()
    assert your_technology_weight <= technology_limit
    assert your_scenario_note.strip()
    print('Rebalance checks passed. Explain why invested weights alone do not add up to 100%.')